# MobileNetV2 Paper Walkthrough: The Smarter Tiny Giant#

Understanding and implementing MobileNetV2 with Pytorch - The next Generation of MobileNet-v1

## Introduction

MobileNetV1 was a breakthrough in the field of computer vision as it proved that deep learning models do not necessarily need to be computationally expensive to achieve high accuracy.

This first version of MobileNet was first proposed back in April 2017 in a paper titled MobileNets: Efficient Convolutional Neural Networks for Mobile Vision Applications

Not long after — in January 2018 to be precise — Sandler et al. from the same institution introduced the successor of MobileNetV1 in a paper titled MobileNetV2: Inverted Residuals and Linear Bottlenecks

which brings significant improvement over the previous one in terms of both accuracy and efficiency. In this article, I am going to walk you through the ideas proposed in the MobileNetV2 paper and show you how to implement the architecture from scratch.

## The Improvements

The first version of MobileNet relies solely on the so-called depthwise separable convolution layers. It is indeed necessary to acknowledge that using these layers as a replacement of standard convolutions allows the model to be extremely lightweight.

However, authors thought that this architecture could still be improved even further. They came up with an idea where instead of only using depthwise separable convolutions, they also adopted the inverted residual and linear bottleneck mechanisms — which is where the title of the MobileNetV2 paper came from.

### Inverted Residual

If you’re familiar with ResNet, I believe you know the so-called bottleneck block. For those who don’t, it is essentially a mechanism where the building block of the network works by following the wide → narrow → wide pattern. Figure 1 below displays the illustration of a bottleneck block used in ResNet. Here we can see that it initially accepts a 256-channel tensor, shrink it to 64, and expands it back to 256.

![](image1.png)

The inverted version of the above block is commonly known as inverted bottleneck, which follows the narrow → wide → narrow structure. Figure 2 below shows an example from the ConvNeXt paper [5], where the number of channels in the input tensor is 96, expanded to 384, and compressed back to 96 by the last convolution layer. It is important to note that in MobileNetV2 an inverted bottleneck block is called inverted residual for some reasons. So, starting from now on, I will use the term to avoid confusion.

![](image2.png)

At this point you might be wondering why we don’t just use the standard bottleneck for MobileNetV2. The answer lies in the original purpose of the standard bottleneck design, where it was first introduced to reduce computational complexity. This was essentially done because ResNet is computationally expensive by nature yet rich in information. For this reason, ResNet authors proposed to reduce computational cost by shrinking the tensor size in the middle of each building block, which is how the bottleneck block was born.

This reduction in the number of channels does not hurt the model capacity that much since ResNet already has a large number of channels overall

On the other hand, MobileNetV2 is intended to be as lightweight as possible in the first place, which means the model capacity is not as high as ResNet

In order to increase model capacity, authors expand the tensor size in the middle to form the inverted residual block, which allows the model to learn more patterns while only slightly increasing complexity

So in short, the middle part of a bottleneck block (narrow) is used for efficiency, while the middle part of an inverted residual block (wide) is used to learn complex patterns. 

If we try to apply a standard bottleneck on MobileNetV2 instead, the computation is going to be even faster, but this might cause a drop in accuracy since the model will lose a significant amount of information.

### Linear Bottleneck

The next concept we need to understand is the so-called linear bottleneck

This one is actually pretty simple since what we essentially do here is just to omit the nonlinearity (i.e., the ReLU activation function) in the last layer of each inverted residual block.

The use of activation functions in neural networks at the first place is to allow the network to capture complex patterns. 

However, it will destroy important information instead if we apply it on a low-dimensional tensor, especially in the context of MobileNetV2 where the inverted residual block projects a high dimensional tensor to a smaller one in the last convolution layer

By removing the activation function in the last convolution layer like this, we essentially prevent the model from losing important information.

Figure 3 below shows what the inverted residual block used in MobileNetV2 looks like. Notice that ReLU is not applied in the last pointwise convolution, which essentially means that this layer behaves somewhat similarly to a standard linear regression layer.

In addition to this figure, the variables k and k’ denote the number of input and output channels, respectively. In the intermediate process, we essentially expand the number of channels by t before eventually shrink it to k’. I’ll go into more detail on these variables in the next section.

![](image3.png)

### ReLU6

So why do we use ReLU6 instead of regular ReLU? In case you’re not yet familiar with it, this activation function is actually similar to ReLU, except that the output value is capped at 6. So, any input greater than 6 will be mapped to that number. Meanwhile, the behavior for negative inputs is exactly the same. Thus, we can simply say that the output of ReLU6 will always be within the range of 0 to 6 (inclusive). Look at Figure 4 below to better understand this idea.

![](image4.png)

In standard ReLU, there is a possibility where the input — and therefore the output — value goes arbitrarily large, in which it potentially causes instability in low-precision environments. 

Remember that MobileNet is intended to be able to work on small devices, in which we know that such devices typically expect small numbers to save memory, say 8-bit integer. In this particular case, having very large activation values could lead to precision loss or clipping when quantized to low-bit representations. Thus, to keep the values small and within a manageable range, we can simply employ ReLU6 to do so.

## The Complete MobileNetV2 Architecture

Now let’s take a look at the complete MobileNetV2 architecture in Figure 5 below. Just like the first version of MobileNet which mostly consists of depthwise separable convolutions, most of the components inside MobileNetV2 are the inverted residual blocks with linear bottlenecks we discussed earlier. Every row in the following table labeled as bottleneck corresponds to a single stage, in which each of them consists of several inverted residual blocks. Talking about the columns in the table, t represents expansion factor used in the middle part of each block, c denotes the number of output channels of each block, n is the number of repeats of the block within that stage, and s indicates the stride of the first block within the stage.

o better understand this idea, let’s take a closer look at the stage which the input shape is 56×56×24. Here you can see that the corresponding parameters of this stage are t=6, c=32, n=3, and s=2. This essentially means that the inverted residual stage consists of 3 blocks. All these blocks are identical except that the first one uses stride 2, reducing the spatial dimension by half from 56×56 to 28×28. Next, c=32 is pretty straightforward as it basically says that the number of output channel of each block within the stage is 32. Meanwhile, t=6 indicates that the intermediate layer inside the blocks is 6 times wider than the input, forming the inverted bottleneck structure. So, in this case the number of channels in the process is going to be 32 → 192 → 32. However, it is important to note that the first block within that stage is different, where it uses 24 → 144 → 32 structure thanks to the 24-channel input tensor. If we refer back to Figure 3, these two structures essentially follow the k → kt → k’ pattern.

![](image5.png)

- Ý nghĩa các cột:

![](image6.png)

- Dòng đầu tiên - Stem:
    - Input: 224² × 3,Operator: conv2d,c = 32, s = 2
    - Conv 3×3 stride 2 (224×224×3 → 112×112×32)
        - Giảm spatial
        - Tăng channel
- Từ dòng 2 trở đi, bottleneck = inverted residual block:
    - Cấu trúc mỗi block: 
        - 1x1 expand (×t)
        - 3x3 depthwise (stride s)
        - 1x1 project (→ c)
        - (+ residual nếu s=1 và channel match)

In addition to the above architecture, here we also have skip-connections placed within the inverted residual blocks. This skip-connection will only be applied whenever the stride of the block is set to 1

This is essentially because the spatial dimension of the image will change whenever we use stride 2, causing the output tensor to have different shape to that of the input

Such a difference in tensor shapes will effectively prevent us from performing element-wise summation between the original flow and the skip-connection. See Figure 6 below for the details. Note that the two illustrations in this figure are basically just the visualization of the table in Figure 3.

![](image7.png)

#### Parameter Tuning

Similar to MobileNetV1, MobileNetV2 also has two adjustable parameters called width multiplier and input resolution. The former is used to adjust the width of the network, while the latter is for changing the resolution of the input image. The architecture you see in Figure 5 is the base configuration, where we set the width multiplier to 1 and the input resolution to 224×224. With these two parameters, we can tune the model to find a sweet spot that balances accuracy and efficiency based on our needs.

We can technically choose arbitrary numbers for the two parameters, but authors already provided several predetermined numbers for their experiments

To the width multiplier, we can use 0.75, 0.5 or 0.35, in which all of them will make the model smaller.

For instance, if we use 0.5 then all numbers in column c in Figure 5 will be reduced to half of their defaults. To the input resolution, we can choose either 192×192, 160×160, 128×128 or 96×96 as a replacement for 224×224 if you want to lower the number of operations during inference.

#### Some Experimental Results

Figure 7 below shows what the experimental results done by the authors look like. Even though MobileNetV1 is considered lightweight already, MobileNetV2 proved that its performance is even better in terms of all metrics compared to its predecessor. However, it is necessary to acknowledge that the base MobileNetV2 is not completely superior to other lightweight models especially when taking into account all aspects at once.

![](image8.png)

In order to achieve even better accuracy, authors also tried to enlarge the model instead by changing the width multiplier to 1.4 for the 224×224 input resolution, which in the above figure corresponds to the result in the last row. Doing this definitely causes the model complexity as well as the computation time to get higher, but in return it allows the model to obtain the highest accuracy. The results in Figure 8 also show the similar thing, where all MobileNetV2 variants completely outperform the MobileNetV1 counterpart, with the largest MobileNetV2 obtaining the highest accuracy among all models.

![](image9.png)